# RAG completo com seus dados (nível produção)

Notebook pronto para adaptar aos seus arquivos reais.

## O que ele entrega
- ingestão de `.txt`, `.md`, `.csv`, `.jsonl`, `.pdf`
- chunking com overlap
- embeddings com `sentence-transformers`
- busca vetorial com `FAISS`
- re-ranking opcional
- geração com LLM
- citações / fontes
- persistência do índice
- debug de retrieval

## 1) Instalação

In [1]:
!pip install -U pip pandas sentence-transformers scikit-learn pypdf openai tqdm python-dotenv
!pip install "numpy==1.26.4" "faiss-cpu==1.8.0"

In [3]:
from __future__ import annotations

import os
import re
import json
import time
import pickle
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
# from tqdm.auto import tqdm

## 2) Configuração

In [5]:
@dataclass
class RAGConfig:
    data_dir: str = "./rag_data"
    index_dir: str = "./rag_index"

    chunk_size: int = 900
    chunk_overlap: int = 180
    min_chunk_chars: int = 120

    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    normalize_embeddings: bool = True

    top_k: int = 8
    top_k_rerank: int = 4
    use_reranker: bool = False
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    llm_model: str = "gpt-4.1-mini"
    temperature: float = 0.1
    max_output_tokens: int = 700

    csv_text_columns_priority: Tuple[str, ...] = (
        "text", "content", "description", "message", "body", "notes"
    )

config = RAGConfig()
config

RAGConfig(data_dir='./rag_data', index_dir='./rag_index', chunk_size=900, chunk_overlap=180, min_chunk_chars=120, embedding_model_name='sentence-transformers/all-MiniLM-L6-v2', normalize_embeddings=True, top_k=8, top_k_rerank=4, use_reranker=False, reranker_model_name='cross-encoder/ms-marco-MiniLM-L-6-v2', llm_model='gpt-4.1-mini', temperature=0.1, max_output_tokens=700, csv_text_columns_priority=('text', 'content', 'description', 'message', 'body', 'notes'))

## 3) Helpers

In [6]:
def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p

def approx_token_count(text: str) -> int:
    return max(1, int(len(text) / 4))

## 4) Loaders

In [7]:
def load_txt_or_md(path: Path) -> List[Dict[str, Any]]:
    text = clean_text(path.read_text(encoding="utf-8", errors="ignore"))
    return [{
        "doc_id": sha1_text(f"{path}:{text[:500]}"),
        "source_path": str(path),
        "source_type": path.suffix.lower().lstrip("."),
        "title": path.stem,
        "text": text,
        "metadata": {"file_name": path.name}
    }]

def detect_csv_text_column(df: pd.DataFrame, priority_cols: Tuple[str, ...]) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for p in priority_cols:
        if p.lower() in lower_map:
            return lower_map[p.lower()]

    object_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not object_cols:
        return None

    scored = []
    for c in object_cols:
        series = df[c].dropna().astype(str)
        if len(series) == 0:
            continue
        avg_len = series.str.len().mean()
        uniq_ratio = series.nunique() / max(1, len(series))
        score = avg_len * 0.8 + uniq_ratio * 50
        scored.append((c, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[0][0] if scored else None

def row_to_text(row: pd.Series, text_col: str) -> str:
    parts = [f"{text_col}: {row.get(text_col, '')}"]
    for c, v in row.items():
        if c == text_col:
            continue
        if pd.isna(v):
            continue
        parts.append(f"{c}: {v}")
    return clean_text("\n".join(parts))

def load_csv(path: Path, config: RAGConfig, text_col: Optional[str] = None) -> List[Dict[str, Any]]:
    df = pd.read_csv(path)
    chosen_col = text_col or detect_csv_text_column(df, config.csv_text_columns_priority)
    if chosen_col is None:
        raise ValueError(f"Não encontrei coluna textual em {path}")

    documents = []
    for idx, row in df.iterrows():
        text = row_to_text(row, chosen_col)
        if len(text.strip()) == 0:
            continue

        metadata = {
            "file_name": path.name,
            "row_index": int(idx),
            "text_column": chosen_col,
        }
        for c in df.columns[:15]:
            val = row.get(c)
            if c == chosen_col or pd.isna(val):
                continue
            metadata[f"col_{c}"] = str(val)[:200]

        documents.append({
            "doc_id": sha1_text(f"{path}:{idx}:{text[:500]}"),
            "source_path": str(path),
            "source_type": "csv_row",
            "title": f"{path.stem} | row {idx}",
            "text": text,
            "metadata": metadata,
        })
    return documents

def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    documents = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for idx, line in enumerate(f):
            if not line.strip():
                continue
            obj = json.loads(line)
            text = clean_text(json.dumps(obj, ensure_ascii=False, indent=2))
            documents.append({
                "doc_id": sha1_text(f"{path}:{idx}:{text[:500]}"),
                "source_path": str(path),
                "source_type": "jsonl",
                "title": f"{path.stem} | item {idx}",
                "text": text,
                "metadata": {"file_name": path.name, "row_index": idx},
            })
    return documents

def load_pdf(path: Path) -> List[Dict[str, Any]]:
    from pypdf import PdfReader
    reader = PdfReader(str(path))
    docs = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = clean_text(page.extract_text() or "")
        if not text:
            continue
        docs.append({
            "doc_id": sha1_text(f"{path}:page:{page_num}:{text[:500]}"),
            "source_path": str(path),
            "source_type": "pdf_page",
            "title": f"{path.stem} | page {page_num}",
            "text": text,
            "metadata": {"file_name": path.name, "page": page_num},
        })
    return docs

def load_documents(config: RAGConfig, csv_text_overrides: Optional[Dict[str, str]] = None) -> List[Dict[str, Any]]:
    root = Path(config.data_dir)
    if not root.exists():
        print(f"Pasta {root} não existe ainda.")
        return []

    documents = []
    csv_text_overrides = csv_text_overrides or {}

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        try:
            suffix = path.suffix.lower()
            if suffix in [".txt", ".md"]:
                documents.extend(load_txt_or_md(path))
            elif suffix == ".csv":
                forced_col = csv_text_overrides.get(path.name)
                documents.extend(load_csv(path, config, forced_col))
            elif suffix == ".jsonl":
                documents.extend(load_jsonl(path))
            elif suffix == ".pdf":
                documents.extend(load_pdf(path))
        except Exception as e:
            print(f"[WARN] Falha ao carregar {path}: {e}")

    return documents

## 5) Chunking

In [8]:
def split_text_with_overlap(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    text = clean_text(text)
    if len(text) <= chunk_size:
        return [text] if text else []

    chunks = []
    start = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk = text[start:end]

        if end < text_len:
            candidates = [
                chunk.rfind("\n\n"),
                chunk.rfind("\n"),
                chunk.rfind(". "),
                chunk.rfind("; "),
                chunk.rfind(", "),
            ]
            best = max(candidates)
            if best > int(chunk_size * 0.6):
                end = start + best + 1
                chunk = text[start:end]

        chunk = clean_text(chunk)
        if chunk:
            chunks.append(chunk)

        if end >= text_len:
            break
        start = max(end - chunk_overlap, start + 1)

    return chunks

def build_chunks(documents: List[Dict[str, Any]], config: RAGConfig) -> List[Dict[str, Any]]:
    chunks = []
    for doc in documents:
        pieces = split_text_with_overlap(doc["text"], config.chunk_size, config.chunk_overlap)
        for i, piece in enumerate(pieces):
            if len(piece) < config.min_chunk_chars:
                continue
            chunks.append({
                "chunk_id": sha1_text(f"{doc['doc_id']}:{i}:{piece[:400]}"),
                "doc_id": doc["doc_id"],
                "source_path": doc["source_path"],
                "source_type": doc["source_type"],
                "title": doc["title"],
                "text": piece,
                "metadata": {
                    **doc.get("metadata", {}),
                    "chunk_index": i,
                    "char_len": len(piece),
                    "approx_tokens": approx_token_count(piece),
                },
            })
    return chunks

In [9]:
documents = load_documents(config)
print(f"Documents carregados: {len(documents)}")

chunks = build_chunks(documents, config)
print(f"Chunks gerados: {len(chunks)}")

if chunks:
    pd.DataFrame([{
        "title": c["title"],
        "source_type": c["source_type"],
        "chars": len(c["text"]),
        "approx_tokens": c["metadata"]["approx_tokens"],
        "source_path": c["source_path"],
    } for c in chunks[:10]])
else:
    print("Coloque seus arquivos em rag_data/ e rode de novo.")

Documents carregados: 13
Chunks gerados: 17


## 6) Embeddings + índice vetorial

In [10]:
from sentence_transformers import SentenceTransformer
import faiss

class VectorIndex:
    def __init__(self, config: RAGConfig):
        self.config = config
        self.model = SentenceTransformer(config.embedding_model_name)
        self.index = None
        self.chunk_store: List[Dict[str, Any]] = []

    def embed(self, texts: List[str], batch_size: int = 64) -> np.ndarray:
        emb = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=self.config.normalize_embeddings,
        )
        return emb.astype("float32")

    def build(self, chunks: List[Dict[str, Any]]) -> None:
        self.chunk_store = chunks
        vectors = self.embed([c["text"] for c in chunks])
        dim = vectors.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(vectors)

    def search(self, query: str, top_k: int = 5, metadata_filter: Optional[Dict[str, Any]] = None) -> List[Dict[str, Any]]:
        if self.index is None:
            raise RuntimeError("Índice ainda não foi construído.")

        q = self.embed([query])
        scores, idxs = self.index.search(q, top_k * 5 if metadata_filter else top_k)

        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1:
                continue
            chunk = self.chunk_store[int(idx)]

            if metadata_filter:
                ok = True
                for k, v in metadata_filter.items():
                    if chunk["metadata"].get(k) != v:
                        ok = False
                        break
                if not ok:
                    continue

            results.append({"score": float(score), "chunk": chunk})
            if len(results) >= top_k:
                break
        return results

    def save(self, path: str | Path) -> None:
        path = ensure_dir(path)
        faiss.write_index(self.index, str(path / "faiss.index"))
        with open(path / "chunk_store.pkl", "wb") as f:
            pickle.dump(self.chunk_store, f)
        with open(path / "config.json", "w", encoding="utf-8") as f:
            json.dump(asdict(self.config), f, ensure_ascii=False, indent=2)

    def load(self, path: str | Path) -> None:
        path = Path(path)
        self.index = faiss.read_index(str(path / "faiss.index"))
        with open(path / "chunk_store.pkl", "rb") as f:
            self.chunk_store = pickle.load(f)

In [11]:
vector_index = VectorIndex(config)

if chunks:
    vector_index.build(chunks)
    print("Índice vetorial criado com sucesso.")
else:
    print("Sem chunks ainda.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14450.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.39it/s]

Índice vetorial criado com sucesso.


## 7) Re-ranking opcional

In [12]:
class OptionalReranker:
    def __init__(self, model_name: str):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder(model_name)

    def rerank(self, query: str, retrieved: List[Dict[str, Any]], top_k: int) -> List[Dict[str, Any]]:
        pairs = [(query, item["chunk"]["text"]) for item in retrieved]
        scores = self.model.predict(pairs)

        rescored = []
        for item, score in zip(retrieved, scores):
            rescored.append({
                "score": float(score),
                "chunk": item["chunk"],
            })
        rescored.sort(key=lambda x: x["score"], reverse=True)
        return rescored[:top_k]

reranker = OptionalReranker(config.reranker_model_name) if config.use_reranker else None
print("Reranker ativado." if reranker else "Reranker desativado.")

Reranker desativado.


## 8) Retrieval e debug

In [13]:
def retrieve_context(
    query: str,
    vector_index: VectorIndex,
    config: RAGConfig,
    metadata_filter: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    retrieved = vector_index.search(query, top_k=config.top_k, metadata_filter=metadata_filter)
    if config.use_reranker and reranker is not None and retrieved:
        retrieved = reranker.rerank(query, retrieved, top_k=config.top_k_rerank)
    else:
        retrieved = retrieved[:config.top_k_rerank]
    return retrieved

def preview_retrieval(query: str, metadata_filter: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    rows = []
    retrieved = retrieve_context(query, vector_index, config, metadata_filter)
    for item in retrieved:
        c = item["chunk"]
        rows.append({
            "score": round(item["score"], 4),
            "title": c["title"],
            "source_path": c["source_path"],
            "source_type": c["source_type"],
            "chunk_index": c["metadata"].get("chunk_index"),
            "page": c["metadata"].get("page"),
            "text_preview": c["text"][:240].replace("\n", " "),
        })
    return pd.DataFrame(rows)

# Exemplo:
# preview_retrieval("qual é a política de reembolso?")

## 9) Prompt grounding

In [14]:
SYSTEM_PROMPT = '''
Você é um assistente de RAG para ambiente corporativo.
Responda apenas com base no contexto fornecido.
Se a resposta não estiver claramente suportada pelo contexto, diga:
"Não encontrei informação suficiente no contexto fornecido."

Regras:
1. Não invente fatos.
2. Seja direto e claro.
3. Sempre cite as fontes usadas no final.
4. Se houver conflito entre fontes, mencione o conflito.
5. Se a pergunta pedir procedimento, responda em passos objetivos.
'''.strip()

def build_context_block(retrieved: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, item in enumerate(retrieved, start=1):
        c = item["chunk"]
        meta = c["metadata"]
        source_ref = f"{Path(c['source_path']).name}"
        if meta.get("page"):
            source_ref += f" | página {meta['page']}"
        source_ref += f" | chunk {meta.get('chunk_index')}"
        blocks.append(
            f"[Fonte {i}]\n"
            f"Título: {c['title']}\n"
            f"Origem: {source_ref}\n"
            f"Conteúdo:\n{c['text']}"
        )
    return "\n\n".join(blocks)

def build_user_prompt(question: str, retrieved: List[Dict[str, Any]]) -> str:
    context_block = build_context_block(retrieved)
    return f'''
Contexto recuperado:
{context_block}

Pergunta do usuário:
{question}

Formato esperado:
- Resposta objetiva
- Se útil, use bullets curtos
- Finalize com: "Fontes usadas:"
'''.strip()

## 10) Chamada do LLM

In [22]:
import dotenv

def generate_answer_llm(
    question: str,
    retrieved: List[Dict[str, Any]],
    config: RAGConfig,
) -> str:
    dotenv.load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return (
            "OPENAI_API_KEY não encontrada.\n\n"
            "O retrieval funcionou, mas a geração não foi executada.\n"
            "Defina a variável de ambiente e rode novamente."
        )

    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    user_prompt = build_user_prompt(question, retrieved)

    response = client.responses.create(
        model=config.llm_model,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=config.temperature,
        max_output_tokens=config.max_output_tokens,
    )
    return response.output_text

## 11) Pipeline final

In [20]:
def ask_rag(
    question: str,
    vector_index: VectorIndex,
    config: RAGConfig,
    metadata_filter: Optional[Dict[str, Any]] = None,
    debug: bool = True,
) -> Dict[str, Any]:
    t0 = time.time()
    retrieved = retrieve_context(question, vector_index, config, metadata_filter)
    retrieval_time = time.time() - t0

    answer = generate_answer_llm(question, retrieved, config)
    total_time = time.time() - t0

    result = {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "metrics": {
            "retrieval_time_sec": round(retrieval_time, 3),
            "total_time_sec": round(total_time, 3),
            "num_context_chunks": len(retrieved),
        }
    }

    if debug:
        print("=== MÉTRICAS ===")
        print(result["metrics"])

        print("\n=== CONTEXTO RECUPERADO ===")
        for i, item in enumerate(retrieved, start=1):
            c = item["chunk"]
            print(f"\n[{i}] score={item['score']:.4f} | {c['title']} | {c['source_path']}")
            print(c["text"][:400].replace("\n", " "))
            print("-" * 90)

        print("\n=== RESPOSTA ===")
        print(answer)

    return result

# Exemplo:
# result = ask_rag("Qual é a política de chargeback?", vector_index, config)

In [23]:
result = ask_rag("Qual é as melhores skills de science?", vector_index, config)

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.99it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 12) Tabela de fontes

In [17]:
def get_sources_table(result: Dict[str, Any]) -> pd.DataFrame:
    rows = []
    for i, item in enumerate(result["retrieved"], start=1):
        c = item["chunk"]
        rows.append({
            "rank": i,
            "score": round(item["score"], 4),
            "title": c["title"],
            "source_path": c["source_path"],
            "source_type": c["source_type"],
            "page": c["metadata"].get("page"),
            "chunk_index": c["metadata"].get("chunk_index"),
            "text_preview": c["text"][:250].replace("\n", " "),
        })
    return pd.DataFrame(rows)

## 13) Dataset fake para testar

Se a pasta `rag_data/` estiver vazia, use isso para validar o pipeline.

In [ ]:
def create_demo_dataset(base_dir: str = "./rag_data_demo") -> str:
    base = ensure_dir(base_dir)

    (base / "policy.md").write_text(
        '''
# Política de Reembolso

Reembolsos podem ser solicitados em até 7 dias corridos após a cobrança.
Após esse prazo, o caso deve ser analisado manualmente pelo time financeiro.
Chargebacks devem ser escalados para o time de risco.
'''.strip(),
        encoding="utf-8"
    )

    pd.DataFrame([
        {"id": 1, "category": "payment_error", "description": "Cartão recusado por saldo insuficiente."},
        {"id": 2, "category": "risk", "description": "Chargeback aberto pelo emissor do cartão."},
        {"id": 3, "category": "support", "description": "Cliente deseja cancelar a assinatura."},
    ]).to_csv(base / "tickets.csv", index=False)

    return str(base)

demo_dir = create_demo_dataset()
demo_dir

In [ ]:
demo_config = RAGConfig(data_dir=demo_dir, index_dir="./rag_index_demo", use_reranker=False)
demo_docs = load_documents(demo_config)
demo_chunks = build_chunks(demo_docs, demo_config)

demo_index = VectorIndex(demo_config)
demo_index.build(demo_chunks)

preview_retrieval("em quanto tempo o cliente pode pedir reembolso?")

## 14) Avaliação rápida do retrieval

In [ ]:
def evaluate_retrieval_hit_rate(
    qa_pairs: List[Dict[str, Any]],
    vector_index: VectorIndex,
    config: RAGConfig,
    expected_source_contains_key: str = "expected_source_contains",
) -> pd.DataFrame:
    rows = []

    for item in qa_pairs:
        q = item["question"]
        expected = item.get(expected_source_contains_key, "")
        retrieved = retrieve_context(q, vector_index, config)
        paths = [r["chunk"]["source_path"] for r in retrieved]
        hit = any(expected.lower() in p.lower() for p in paths)

        rows.append({
            "question": q,
            "expected_source_contains": expected,
            "hit": hit,
            "top_sources": " | ".join(Path(p).name for p in paths[:3]),
        })

    return pd.DataFrame(rows)

qa_pairs_demo = [
    {"question": "qual o prazo para pedir reembolso?", "expected_source_contains": "policy.md"},
    {"question": "o que fazer em caso de chargeback?", "expected_source_contains": "policy.md"},
    {"question": "qual ticket fala de saldo insuficiente?", "expected_source_contains": "tickets.csv"},
]

evaluate_retrieval_hit_rate(qa_pairs_demo, demo_index, demo_config)

## 15) Checklist de produção

- persistir índice em disco
- guardar metadados ricos
- logar query + chunks retornados
- medir retrieval separado da resposta
- limitar contexto
- usar filtros por tipo, data ou origem
- adicionar cache
- monitorar latência e custo
- evoluir para hybrid search
- adicionar re-ranking de verdade

## 16) Como adaptar para seus dados

### Docs em pasta
Jogue os arquivos em `rag_data/`.

### CSV de tickets / erros
Garanta uma coluna como:
- `description`
- `message`
- `text`
- `content`

Se não bater, force manualmente:
```python
documents = load_documents(
    config,
    csv_text_overrides={"seu_arquivo.csv": "nome_da_coluna"}
)
```

### Filtro por arquivo
```python
ask_rag(
    "qual a regra de reembolso?",
    vector_index,
    config,
    metadata_filter={"file_name": "policy.md"}
)
```

## 17) Resumo honesto

Se o RAG estiver ruim, normalmente o problema está em:
- chunking
- embeddings
- retrieval
- contexto poluído

Quase nunca vale culpar o LLM primeiro.

**Workflow profissional: debuga retrieval primeiro, depois a geração.**